In [ ]:
!pip -q install openai pymupdf faiss-cpu numpy

In [ ]:
import os
import time
import re
from typing import List

import numpy as np
import faiss
import fitz
from google.colab import files

from openai import OpenAI

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "PASTE_YOUR_KEY_HERE")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]
print("Uploaded:", pdf_file)

def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(len(doc)):
        pages.append(doc.load_page(i).get_text())
    return "\n".join(pages)

raw_text = extract_text_from_pdf(pdf_file).strip()
print("Extracted characters:", len(raw_text))
print(raw_text[:600])

In [ ]:
def _split_by_separator(text: str, sep: str) -> List[str]:
    if sep == "":
        return list(text)
    parts = text.split(sep)
    out = []
    for i, p in enumerate(parts):
        out.append(p + sep if i < len(parts) - 1 else p)
    return out

def recursive_text_splitter(
    text: str,
    chunk_size: int = 900,
    chunk_overlap: int = 120,
    separators: List[str] = None,
    min_chunk_chars: int = 200
) -> List[str]:
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    text = (text or "").strip()
    if not text:
        return []

    def _recurse(t: str, seps: List[str]) -> List[str]:
        if len(t) <= chunk_size or not seps:
            return [t]

        sep = seps[0]
        parts = _split_by_separator(t, sep)
        if len(parts) == 1:
            return _recurse(t, seps[1:])

        out, buf = [], ""
        for part in parts:
            if len(part) > chunk_size and seps[1:]:
                if buf.strip():
                    out.append(buf)
                    buf = ""
                out.extend(_recurse(part, seps[1:]))
                continue

            if len(buf) + len(part) <= chunk_size:
                buf += part
            else:
                if buf.strip():
                    out.append(buf)
                buf = part

        if buf.strip():
            out.append(buf)
        return out

    pieces = _recurse(text, separators)

    cleaned = []
    for p in pieces:
        p = p.strip()
        if not p:
            continue
        if cleaned and len(p) < min_chunk_chars:
            cleaned[-1] = (cleaned[-1].rstrip() + " " + p).strip()
        else:
            cleaned.append(p)

    chunks = []
    for p in cleaned:
        if not chunks:
            chunks.append(p)
            continue
        overlap_text = chunks[-1][-chunk_overlap:] if chunk_overlap > 0 else ""
        merged = (overlap_text + p).strip()
        if len(merged) > chunk_size:
            merged = merged[-chunk_size:]
        chunks.append(merged)

    return chunks

chunks = recursive_text_splitter(raw_text, chunk_size=900, chunk_overlap=120)
print("Chunks:", len(chunks))
print("Sample chunk:\n", chunks[0][:500])

In [ ]:
EMBED_MODEL = "text-embedding-3-small"

def embed_texts(texts: List[str], batch_size: int = 64) -> np.ndarray:
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
    return np.array(vectors, dtype=np.float32)

t0 = time.time()
chunk_vecs = embed_texts(chunks, batch_size=64)
t1 = time.time()

dim = chunk_vecs.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(chunk_vecs)

print("Embeddings shape:", chunk_vecs.shape)
print("FAISS vectors:", index.ntotal)
print("Embedding time (s):", round(t1 - t0, 2))

In [ ]:
def retrieve(question: str, top_k: int = 3):
    q_vec = embed_texts([question], batch_size=1)
    distances, indices = index.search(q_vec, top_k)
    hits = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        hits.append({"rank": rank, "idx": int(idx), "dist": float(dist), "text": chunks[int(idx)]})
    return hits

In [ ]:
GEN_MODEL = "gpt-4o-mini"

def ask_question(question: str, top_k: int = 3) -> str:
    hits = retrieve(question, top_k=top_k)

    context_blocks = []
    for h in hits:
        context_blocks.append(f"[Source {h['rank']}]\n{h['text']}")

    context = "\n\n".join(context_blocks)

    prompt = (
        "You are a helpful assistant.\n"
        "Use ONLY the context to answer.\n"
        "If the answer is not in the context, say: I do not know.\n"
        "Write a clean answer in 1 to 3 sentences.\n"
        "Do not include numbering or bullet points.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    resp = client.responses.create(
        model=GEN_MODEL,
        input=prompt
    )

    answer = resp.output_text.strip()
    answer = re.sub(r"^\s*\(?\d+\)?[.)]\ s+", "", answer).strip()
    return answer

In [ ]:
print(ask_question("What are the benefits?", top_k=3))

In [ ]:
while True:
    query = input("\nAsk a question (type 'exit' to stop): ").strip()
    if query.lower() == "exit":
        break

    print("\nAnswer:\n")
    print(ask_question(query, top_k=3))